# Phase 4: Integration — LLM Mapping → Step 3 no_match primary_tag

Assign Phase 3 output (58,980 unique title → onet_tag) to Step 3 output (123,849 rows)
`no_match` rows `primary_tag`.

- **Input 1**: `step3_with_primary_tag/` (123,849 × 7)
- **Input 2**: `phase3_llm_mapping/output/no_match_onet_mapped.parquet` (58,980 × 2)
- **Output**: `phase4_integration/step4_with_llm_primary_tag.parquet` (123,849 × 7)

In [1]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path

STEP3 = Path('../../data/processed/step3_with_primary_tag')
PHASE3 = Path('../../data/processed/step4_llm_onet_normalization/phase3_llm_mapping/output/no_match_onet_mapped.parquet')
OUTPUT_DIR = Path('../../data/processed/step4_llm_onet_normalization/phase4_integration')

# Load Step 3 output (pyarrow direct — Python 3.14 compatibility)
step3 = pq.read_table(STEP3).to_pandas()

print(f'Step 3 rows: {len(step3):,}')
print(f'Columns: {list(step3.columns)}')
print(f'\nMatch type distribution:')
print(step3['match_type'].value_counts())
print(f'\nprimary_tag null count: {step3["primary_tag"].isna().sum():,}')

Step 3 rows: 123,849
Columns: ['id', 'title', 'description', 'seniority', 'seniority_removed_title', 'primary_tag', 'match_type']

Match type distribution:
match_type
no_match      102322
partial        14452
exact_main      4979
exact_sub       2096
Name: count, dtype: int64

primary_tag null count: 0


In [2]:
# Load Phase 3 output (unique title → onet_tag lookup)
phase3 = pq.read_table(PHASE3).to_pandas()

print(f'Phase 3 rows: {len(phase3):,}')
print(f'Columns: {list(phase3.columns)}')
print(f'\nonet_tag distribution:')
mapped = phase3['onet_tag'].notna() & (phase3['onet_tag'] != 'error')
null_tag = phase3['onet_tag'].isna()
error_tag = phase3['onet_tag'] == 'error'
print(f'  Mapped : {mapped.sum():,}')
print(f'  Null   : {null_tag.sum():,}')
print(f'  Error  : {error_tag.sum():,}')

Phase 3 rows: 58,980
Columns: ['seniority_removed_title', 'onet_tag']

onet_tag distribution:
  Mapped : 57,317
  Null   : 1,425
  Error  : 238


In [3]:
# Merge: Assign Phase 3 onet_tag → primary_tag for no_match rows
no_match_mask = step3['match_type'] == 'no_match'

# Matched rows: keep as-is
matched = step3[~no_match_mask].copy()

# No_match rows: LEFT JOIN with Phase 3 lookup
no_match = step3[no_match_mask].drop(columns=['primary_tag']).merge(
    phase3.rename(columns={'onet_tag': 'primary_tag'}),
    on='seniority_removed_title',
    how='left'
)

# Combine
result = pd.concat([matched, no_match], ignore_index=True)

print(f'Result rows: {len(result):,} (expected: {len(step3):,})')
print(f'Columns: {list(result.columns)}')

Result rows: 123,849 (expected: 123,849)
Columns: ['id', 'title', 'description', 'seniority', 'seniority_removed_title', 'primary_tag', 'match_type']


In [4]:
# Verification
print('=== Verification ===')
print(f'\n1. Row count: {len(result):,} (expected 123,849)')
assert len(result) == len(step3), f'Row count mismatch: {len(result)} != {len(step3)}'

print(f'\n2. Match type distribution (should be unchanged):')
print(result['match_type'].value_counts())

print(f'\n3. primary_tag status (no_match rows only):')
no_match_result = result[result['match_type'] == 'no_match']
filled = no_match_result['primary_tag'].notna()
error_filled = no_match_result['primary_tag'] == 'error'
print(f'  Total no_match  : {len(no_match_result):,}')
print(f'  primary_tag set : {filled.sum():,} ({filled.sum()/len(no_match_result)*100:.1f}%)')
print(f'  primary_tag null: {(~filled).sum():,} ({(~filled).sum()/len(no_match_result)*100:.1f}%)')
print(f'  primary_tag error: {error_filled.sum():,}')

print(f'\n4. Matched rows primary_tag unchanged:')
matched_result = result[result['match_type'] != 'no_match']
print(f'  Matched rows: {len(matched_result):,}')
print(f'  primary_tag null: {matched_result["primary_tag"].isna().sum()}')

print(f'\n5. Overall primary_tag coverage:')
total_filled = result['primary_tag'].notna()
print(f'  Filled: {total_filled.sum():,} / {len(result):,} ({total_filled.sum()/len(result)*100:.1f}%)')
print(f'  Null  : {(~total_filled).sum():,}')

=== Verification ===

1. Row count: 123,849 (expected 123,849)

2. Match type distribution (should be unchanged):
match_type
no_match      102322
partial        14452
exact_main      4979
exact_sub       2096
Name: count, dtype: int64

3. primary_tag status (no_match rows only):
  Total no_match  : 102,322
  primary_tag set : 99,594 (97.3%)
  primary_tag null: 2,728 (2.7%)
  primary_tag error: 489

4. Matched rows primary_tag unchanged:
  Matched rows: 21,527
  primary_tag null: 0

5. Overall primary_tag coverage:
  Filled: 121,121 / 123,849 (97.8%)
  Null  : 2,728


In [5]:
# Save
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / 'step4_with_llm_primary_tag.parquet'

table_out = pa.Table.from_pandas(result)
pq.write_table(table_out, output_path)

# Verify saved file
verify = pq.read_table(output_path).to_pandas()
print(f'Saved: {output_path}')
print(f'Rows : {len(verify):,}')
print(f'Columns: {list(verify.columns)}')
print(f'\nSample (no_match with primary_tag):')
verify[verify['match_type'] == 'no_match'].dropna(subset=['primary_tag']).head(10)

Saved: ..\..\data\processed\step4_llm_onet_normalization\phase4_integration\step4_with_llm_primary_tag.parquet
Rows : 123,849
Columns: ['id', 'title', 'description', 'seniority', 'seniority_removed_title', 'primary_tag', 'match_type']

Sample (no_match with primary_tag):


,id,title,description,seniority,seniority_removed_title,primary_tag,match_type
21527,921716,Marketing Coordinator,Job descriptionA leading real estate firm in N...,not_specified,Marketing Coordinator,Market Research Analysts and Marketing Special...,no_match
21528,1829192,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",not_specified,Mental Health Therapist/Counselor,Mental Health Counselors,no_match
21529,10998357,Assitant Restaurant Manager,The National Exemplar is accepting application...,not_specified,Assitant Restaurant Manager,Food Service Managers,no_match
21530,23221523,Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,senior,Elder Law / Trusts and Estates Associate Attorney,Lawyers,no_match
21531,35982263,Service Technician,Looking for HVAC service tech with experience ...,not_specified,Service Technician,"Maintenance and Repair Workers, General",no_match
21532,91700727,Economic Development and Planning Intern,Job summary:The Economic Development & Plannin...,entry_level,Economic Development and Planning,Urban and Regional Planners,no_match
21533,103254301,Producer,Company DescriptionRaw Cereal is a creative de...,not_specified,Producer,Producers and Directors,no_match
21534,112576855,Building Engineer,Summary: Due to the pending retirement of our ...,not_specified,Building Engineer,Stationary Engineers and Boiler Operators,no_match
21535,2264355,Worship Leader,It is an exciting time to be a part of our chu...,not_specified,Worship Leader,Clergy,no_match
21536,9615617,Inside Customer Service Associate,Glastender Inc. is a family-owned manufacturer...,entry_level,Inside Customer Service,Customer Service Representatives,no_match
